# Homework 1
**Name:** Taha Furkan Torun
**Student ID:** 2022402090

# QUESTION 1 (Spam / Nonspam)

## Question 1(a)

The dataset is split into 80% training and 20% test sets using seed 425. 
Below are the spam percentages for each subset.

In [16]:
install.packages("kernlab")

library(kernlab)

data("spam", package = "kernlab")

set.seed(425)

n <- nrow(spam)
train_index <- sample(1:n, size = 0.8 * n)

train_data <- spam[train_index, ]
test_data  <- spam[-train_index, ]


cat("Overall spam percentage:", mean(spam$type == "spam") * 100, "\n")
cat("Train spam percentage:", mean(train_data$type == "spam") * 100, "\n")
cat("Test spam percentage:", mean(test_data$type == "spam") * 100)





The downloaded binary packages are in
	/var/folders/b5/zg0ny5lj36n2kvl4x5b72p0m0000gn/T//RtmpJj05St/downloaded_packages
Overall spam percentage: 39.40448 
Train spam percentage: 38.42391 
Test spam percentage: 43.32248

## Question 1(b)

Using the training set, the largest possible classification tree is built with the `rpart` package.  
Then, the number of leaf nodes in the tree is reported.

In [17]:
library(rpart)

largest_tree <- rpart(
  type ~ .,
  data = train_data,
  method = "class",
  control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 10)
)

leaf_count <- sum(largest_tree$frame$var == "<leaf>")

cat("Number of leaf nodes:", leaf_count, "\n")

Number of leaf nodes: 252 


## Question 1(c)

Predictions are made on the test set using the largest tree. The error rate, false positive rate, and false negative rate are reported.

In [18]:
pred <- predict(largest_tree, test_data, type = "class")

conf_mat <- table(Predicted = pred, Actual = test_data$type)

conf_mat

test_error <- mean(pred != test_data$type)

TP <- conf_mat["spam", "spam"]
TN <- conf_mat["nonspam", "nonspam"]
FP <- conf_mat["spam", "nonspam"]
FN <- conf_mat["nonspam", "spam"]

false_positive_rate <- FP / (FP + TN)
false_negative_rate <- FN / (FN + TP)

cat("Test error:", round(test_error, 4), "\n")
cat("False positive rate:", round(false_positive_rate, 4), "\n")
cat("False negative rate:", round(false_negative_rate, 4), "\n")

         Actual
Predicted nonspam spam
  nonspam     486   42
  spam         36  357

Test error: 0.0847 
False positive rate: 0.069 
False negative rate: 0.1053 


## Question 1(d)

The tree size that minimizes the cross-validation error is identified using the `rpart` output. Then, using the 1-SE rule, the smallest tree whose cross-validation error is less than or equal to the minimum cross-validation error plus one standard deviation is selected. This tree is called `opttree`.

In [19]:
cp_table <- largest_tree$cptable

min_idx <- which.min(cp_table[, "xerror"])
min_cp <- cp_table[min_idx, "CP"]
min_splits <- cp_table[min_idx, "nsplit"]
min_leaves <- min_splits + 1

threshold <- cp_table[min_idx, "xerror"] + cp_table[min_idx, "xstd"]

opt_idx <- which(cp_table[, "xerror"] <= threshold)[1]
opt_cp <- cp_table[opt_idx, "CP"]
opt_splits <- cp_table[opt_idx, "nsplit"]
opt_leaves <- opt_splits + 1

opttree <- prune(largest_tree, cp = opt_cp)

cat("Tree size with minimum CV error (leaf nodes):", min_leaves, "\n")
cat("Tree size with minimum CV error (splits):", min_splits, "\n")
cat("CP for minimum CV error tree:", min_cp, "\n")

cat("Tree size of opttree (leaf nodes):", opt_leaves, "\n")
cat("Tree size of opttree (splits):", opt_splits, "\n")
cat("CP for opttree:", opt_cp, "\n")

Tree size with minimum CV error (leaf nodes): 44 
Tree size with minimum CV error (splits): 43 
CP for minimum CV error tree: 0.001414427 
Tree size of opttree (leaf nodes): 37 
Tree size of opttree (splits): 36 
CP for opttree: 0.001591231 


## Question 1(e)

Predictions are made on the test set using `opttree`. The error rate, false positive rate, and false negative rate are reported and compared with the results in part (c).

In [20]:
pred_opt <- predict(opttree, test_data, type = "class")

conf_mat_opt <- table(Predicted = pred_opt, Actual = test_data$type)

conf_mat_opt

test_error_opt <- mean(pred_opt != test_data$type)

TP_opt <- conf_mat_opt["spam", "spam"]
TN_opt <- conf_mat_opt["nonspam", "nonspam"]
FP_opt <- conf_mat_opt["spam", "nonspam"]
FN_opt <- conf_mat_opt["nonspam", "spam"]

false_positive_rate_opt <- FP_opt / (FP_opt + TN_opt)
false_negative_rate_opt <- FN_opt / (FN_opt + TP_opt)

cat("Test error:", round(test_error_opt, 4), "\n")
cat("False positive rate:", round(false_positive_rate_opt, 4), "\n")
cat("False negative rate:", round(false_negative_rate_opt, 4), "\n")

         Actual
Predicted nonspam spam
  nonspam     491   46
  spam         31  353

Test error: 0.0836 
False positive rate: 0.0594 
False negative rate: 0.1153 


The opttree has a slightly lower test error compared to the largest tree. It also has a lower false positive rate, but a higher false negative rate. This indicates that the pruned tree is more conservative in predicting spam, resulting in fewer false alarms but missing more actual spam cases.

## QUESTION 2 (Suicide-rate)

In [21]:
list.files()
suicide_data <- read.csv("suicide-rate.csv")
head(suicide_data)
str(suicide_data)

[1] "hw1.ipynb"        "suicide-rate.csv"

,country,year,sex,age,suicides_no,population,suicides.100k.pop,country.year,gdp_for_year....,gdp_per_capita....,generation
,<chr>,<int>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<int>,<chr>
1,Albania,1987,male,15-24 years,21,312900,6.71,Albania1987,"2,156,624,900",796,Generation X
2,Albania,1987,male,35-54 years,16,308000,5.19,Albania1987,"2,156,624,900",796,Silent
3,Albania,1987,female,15-24 years,14,289700,4.83,Albania1987,"2,156,624,900",796,Generation X
4,Albania,1987,male,75+ years,1,21800,4.59,Albania1987,"2,156,624,900",796,G.I. Generation
5,Albania,1987,male,25-34 years,9,274300,3.28,Albania1987,"2,156,624,900",796,Boomers
6,Albania,1987,female,75+ years,1,35600,2.81,Albania1987,"2,156,624,900",796,G.I. Generation


'data.frame':	27820 obs. of  11 variables:
 $ country           : chr  "Albania" "Albania" "Albania" "Albania" ...
 $ year              : int  1987 1987 1987 1987 1987 1987 1987 1987 1987 1987 ...
 $ sex               : chr  "male" "male" "female" "male" ...
 $ age               : chr  "15-24 years" "35-54 years" "15-24 years" "75+ years" ...
 $ suicides_no       : int  21 16 14 1 9 1 6 4 1 0 ...
 $ population        : int  312900 308000 289700 21800 274300 35600 278800 257200 137500 311000 ...
 $ suicides.100k.pop : num  6.71 5.19 4.83 4.59 3.28 2.81 2.15 1.56 0.73 0 ...
 $ country.year      : chr  "Albania1987" "Albania1987" "Albania1987" "Albania1987" ...
 $ gdp_for_year....  : chr  "2,156,624,900" "2,156,624,900" "2,156,624,900" "2,156,624,900" ...
 $ gdp_per_capita....: int  796 796 796 796 796 796 796 796 796 796 ...
 $ generation        : chr  "Generation X" "Silent" "Generation X" "G.I. Generation" ...


In [22]:
suicide_data$gdp_for_year <- as.numeric(gsub(",", "", suicide_data$gdp_for_year))
suicide_data2 <- suicide_data[, !(names(suicide_data) %in% c("country.year"))]

## Question 2(a)

The dataset is partitioned into training and test sets using 80% of the data for training and a seed value of 425.

In [23]:
set.seed(425)

n <- nrow(suicide_data2)
train_idx <- sample(1:n, size = 0.8 * n)

train_data2 <- suicide_data2[train_idx, ]
test_data2  <- suicide_data2[-train_idx, ]
cat("Training set size:", nrow(train_data2), "\n")
cat("Test set size:", nrow(test_data2), "\n")

Training set size: 22256 
Test set size: 5564 


## Question 2(b)

A regression tree is built using the training data, and the tree size that minimizes the cross-validation error is reported.

In [24]:
library(rpart)

tree2 <- rpart(
  suicides.100k.pop ~ .,
  data = train_data2,
  method = "anova",
  control = rpart.control(cp = 0)
)

cp_table2 <- tree2$cptable

min_idx2 <- which.min(cp_table2[, "xerror"])

min_splits2 <- cp_table2[min_idx2, "nsplit"]
min_leaves2 <- min_splits2 + 1

cat("Number of splits:", min_splits2, "\n")
cat("Number of leaf nodes:", min_leaves2, "\n")

Number of splits: 1466 
Number of leaf nodes: 1467 


## Question 2(c)

The importance of input variables is examined based on the fitted regression tree.

In [25]:
tree2$variable.importance

gdp_for_year....                age            country                sex 
        3886717.30         2472647.21         2169463.07         1623498.00 
        generation        suicides_no         population       gdp_for_year 
        1299269.17         1227445.43          380744.09          238810.72 
gdp_per_capita....               year 
         231830.98           46002.67

The most important variables in predicting the suicide rate are GDP for year, age, country, and sex. These variables have the highest importance values in the regression tree, indicating that they contribute the most to the prediction. 

This result is reasonable, as economic conditions (GDP) and demographic factors such as age and sex are likely to have a strong impact on suicide rates. Additionally, the importance of the country variable suggests that there are significant differences across countries, possibly due to cultural or socioeconomic factors.

## Question 2(d)

The cross-validation error is compared with the test error to evaluate whether it is a good predictor.

In [26]:
pred2 <- predict(tree2, test_data2)

test_mse <- mean((pred2 - test_data2$suicides.100k.pop)^2)

min_xerror2 <- cp_table2[min_idx2, "xerror"]

cat("Test MSE:", test_mse, "\n")
cat("Minimum CV error:", min_xerror2, "\n")

Test MSE: 68.2222 
Minimum CV error: 0.2084494 


Although the CV error and test error are measured on different scales, their values suggest similar model performance. This indicates that cross-validation error is a reasonable predictor of test error.